<a href="https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys
import pandas as pd
import numpy as np
import duckdb

REPO_URL = "https://github.com/vahagngrigoryan2006/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"

if not os.path.isdir(REPO_DIR):
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

con = duckdb.connect()

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Set HF_TOKEN as a Colab secret (or env var if running locally) before continuing."

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN \'{HF_TOKEN}\')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT     = f"read_parquet(\'{WAREHOUSE}/dim_content.parquet\')"
FACT_APRIL_MAY  = (
    f"read_parquet([\'{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet\', "
    f"\'{WAREHOUSE}/fact_content_daily_performance/month=2026-04/*.parquet\', "
    f"\'{WAREHOUSE}/fact_content_daily_performance/month=2026-05/*.parquet\'])"
)

# Same window as w04_baseline_score.ipynb: as-of 2026-05-31, decision moment 2026-05-01.
features = con.sql(f"""
    WITH bounds AS (
        SELECT DATE \'2026-05-31\' AS as_of_date
    ),
    per_item AS (
        SELECT f.client_hash_id, f.content_hash_id,
               MIN(f.report_date) AS first_seen,
               SUM(CASE WHEN f.report_date >= b.as_of_date - INTERVAL 30 DAY
                        THEN f.gsc_impressions ELSE 0 END) AS last_30_impressions,
               SUM(CASE WHEN f.report_date <  b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_impressions ELSE 0 END) AS prev_30_impressions,
               SUM(CASE WHEN f.report_date <  b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_clicks ELSE 0 END)      AS prev_30_clicks,
               AVG(CASE WHEN f.report_date <  b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_avg_position END)       AS prev_30_avg_position
        FROM {FACT_APRIL_MAY} f, bounds b
        GROUP BY 1, 2
    )
    SELECT p.*, d.content_created_date, d.content_updated_date
    FROM per_item p
    JOIN {DIM_CONTENT} d USING (content_hash_id)
    WHERE p.first_seen <= DATE \'2026-05-31\' - INTERVAL 60 DAY   -- guard (a): full prev_30 history
      AND p.prev_30_impressions >= 100                             -- guard (b): activity floor
""").df()

decision_moment = pd.Timestamp("2026-05-01")
features["content_created_date"] = pd.to_datetime(features["content_created_date"])
features["content_updated_date"] = pd.to_datetime(features["content_updated_date"])
features["content_age_days_at_decision"] = (decision_moment - features["content_created_date"]).dt.days
features["days_since_last_update_at_decision"] = (decision_moment - features["content_updated_date"]).dt.days
features = features[features["days_since_last_update_at_decision"] > 0]   # guard (c)

features["impressions_pct_change"] = (
    (features["last_30_impressions"] - features["prev_30_impressions"]) / features["prev_30_impressions"]
)
features["is_declining"] = (features["impressions_pct_change"] < -0.20).astype(int)
features["prev_30_ctr"] = features["prev_30_clicks"] / features["prev_30_impressions"] * 100

df = features.copy()
print(f"Working frame: {len(df):,} content items surviving all three guards.")
print(f"is_declining rate: {df['is_declining'].mean():.3f}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Working frame: 18,918 content items surviving all three guards.
is_declining rate: 0.543


In [2]:
key_fields = ["prev_30_impressions", "prev_30_clicks", "prev_30_ctr", "prev_30_avg_position",
              "content_age_days_at_decision", "days_since_last_update_at_decision"]

desc = df[key_fields].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T
desc["max_over_p99"] = (desc["max"] / desc["99%"]).round(1)   # a quick, blunt heavy-tail signal
print("Distributions of key fields:")
desc.round(2)

Distributions of key fields:


,count,mean,std,min,50%,90%,95%,99%,max,max_over_p99
prev_30_impressions,18918.0,1570.56,4899.26,100.00,498.00,2986.30,5828.75,19693.80,267960.00,13.6
prev_30_clicks,18918.0,3.56,15.13,0.00,0.00,7.00,15.00,57.00,1024.00,18.0
prev_30_ctr,18918.0,0.19,0.37,0.00,0.00,0.59,0.85,1.62,10.09,6.2
prev_30_avg_position,18918.0,18.64,15.73,0.14,13.37,40.03,51.63,71.59,89.44,1.2
content_age_days_at_decision,18918.0,217.92,78.30,56.00,219.00,308.00,323.00,381.00,406.00,1.1
days_since_last_update_at_decision,18918.0,59.23,20.80,4.00,65.00,65.00,65.00,67.00,295.00,4.4


In [3]:
zero_ctr_share = (df["prev_30_ctr"] == 0).mean()
print(f"Share of rows with EXACTLY zero prev_30 clicks (prev_30_ctr == 0): {zero_ctr_share:.3f}")
print("This is the number that motivated my change from `ctr < 0.5x tier average` to")
print("`ctr == 0` in w04_baseline_score.ipynb")
print()
print("Heavy-tail read: compare `max` to `99%` in the table above for prev_30_impressions and")
print("prev_30_clicks -- a `max_over_p99` well above 1 means a handful of extreme rows sit far")
print("beyond where the bulk of the data lives (the classic heavy-tail shape for traffic data).")

Share of rows with EXACTLY zero prev_30 clicks (prev_30_ctr == 0): 0.524
This is the number that motivated my change from `ctr < 0.5x tier average` to
`ctr == 0` in w04_baseline_score.ipynb

Heavy-tail read: compare `max` to `99%` in the table above for prev_30_impressions and
prev_30_clicks -- a `max_over_p99` well above 1 means a handful of extreme rows sit far
beyond where the bulk of the data lives (the classic heavy-tail shape for traffic data).


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Test 1 and test 2 mirror what `w04_baseline_score.ipynb` already found — rerun here for a
self-contained notebook, same tier boundaries, same result expected. Test 3 (staleness) is new:
the one canonical flag-linked signal not yet checked on real warehouse data, using the same
`0-3mo / 3-6mo / 6-12mo / 12+mo` buckets and "share declining" framing the W4 deck's own worked
example uses for the refresh flag.

### Test 1 — CTR vs. position (behind the CTR-fix / snippet-review logic)

Same `position_tier` boundaries as `w04_baseline_score.ipynb`: `top_3` ≤3, `page_1` ≤10,
`striking_distance` ≤20, `page_2_3` ≤50, `deep` >50.

**Already confirmed in `w04_baseline_score.ipynb`: CONFIRMED — clear, roughly monotonic drop in
mean CTR as position worsens, smallest tier `n` = 664.**

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

position_bins = [0, 3, 10, 20, 50, np.inf]
position_labels = ["top_3", "page_1", "striking_distance", "page_2_3", "deep"]
df["position_tier"] = pd.cut(df["prev_30_avg_position"], bins=position_bins, labels=position_labels)

test1_table = df.groupby("position_tier", observed=True)["prev_30_ctr"].agg(["mean", "count"]).round(3)
test1_table.columns = ["mean_ctr_pct", "n"]
print("Test 1 -- mean prev_30 CTR (%) by position tier:")
test1_table

Test 1 -- mean prev_30 CTR (%) by position tier:


,mean_ctr_pct,n
position_tier,,
top_3,0.399,664
page_1,0.261,6959
striking_distance,0.203,4519
page_2_3,0.107,5752
deep,0.035,1024


### Test 2 — Volume vs. CTR (behind quick-win logic)

Same `volume_tier` boundaries as `w04_baseline_score.ipynb`, anchored on the rule's own `>= 500`
eligibility cutoff.

**Already confirmed in `w04_baseline_score.ipynb`: CONFIRMED — mean CTR rises monotonically with
volume, smallest tier `n` = 1,094.**

In [5]:
volume_bins = [0, 500, 2000, 5000, np.inf]
volume_labels = ["thin_lt500", "moderate_500_2000", "healthy_2000_5000", "high_5000plus"]
df["volume_tier"] = pd.cut(df["prev_30_impressions"], bins=volume_bins, labels=volume_labels)

test2_table = df.groupby("volume_tier", observed=True)["prev_30_ctr"].agg(["mean", "count"]).round(3)
test2_table.columns = ["mean_ctr_pct", "n"]
print("Test 2 -- mean prev_30 CTR (%) by volume tier:")
test2_table

Test 2 -- mean prev_30 CTR (%) by volume tier:


,mean_ctr_pct,n
volume_tier,,
thin_lt500,0.159,9499
moderate_500_2000,0.222,6641
healthy_2000_5000,0.238,1684
high_5000plus,0.247,1094


### Test 3 — Staleness vs. CTR

We use `days_since_last_update_at_decision` for eligibility (we do not consider rows with `days_since_last_update_at_decision <= 60`). Checked against CTR: does staleness itself carry information, or is it "just" a reliability gate?

**Conclusion**: OPPOSITE/MIXED: We would normally expect ctr to fall as the last updated date goes far. But here we see a rise from 2nd to 4th categories. Note that here `n` is not comfortable large for some tiers. So we treat staleness as just an eligibility gate, not a valuable signal.

In [6]:
staleness_bins = [0, 60, 120, 180, 365, np.inf]
staleness_labels = ["0_2mo", "2_4mo", "4_6mo", "6_12mo", "12mo_plus"]
df["staleness_bucket"] = pd.cut(df["days_since_last_update_at_decision"], bins=staleness_bins, labels=staleness_labels)

test3_table = df.groupby("staleness_bucket", observed=True)["prev_30_ctr"].agg(["mean", "count"]).round(3)
test3_table.columns = ["mean_ctr_pct", "n"]
print("Test 3 -- mean CTR by staleness bucket (deck's exact bucket shape, real data):")
test3_table

Test 3 -- mean CTR by staleness bucket (deck's exact bucket shape, real data):


,mean_ctr_pct,n
staleness_bucket,,
0_2mo,0.221,2400
2_4mo,0.188,16364
4_6mo,0.280,131
6_12mo,0.251,23


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

This tests the **exact, specific assumption `w04_baseline_score.ipynb`'s live rule makes**. That rule flags anything with `prev_30_ctr == 0` (among eligible rows) as `CTR_BELOW_POSITION_PEERS`. The assumption baked into treating "zero clicks" as meaningful evidence is that zero-clicks is *rare*,  a real anomaly worth a human's time.

**Conclusion**: The first three tiers satisfy, so we can treat zeros as an anomaly.
The last tier's majority (`67.5%`) has `prev_30_ctr = 0`, which means that in *deep* positions it is normal to have no clicks at all. However, given that all of them still have enough impression, we still flag them.




In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

enough_impressions = df["prev_30_impressions"] >= 500
not_brand_new       = df["content_age_days_at_decision"] >= 90
not_in_cooldown      = df["days_since_last_update_at_decision"] >= 60
eligible = enough_impressions & not_brand_new & not_in_cooldown

df_eligible = df[eligible].copy()
df_eligible["zero_click"] = (df_eligible["prev_30_ctr"] == 0)

flag_test_table = df_eligible.groupby("position_tier", observed=True)["zero_click"].agg(["mean", "count"]).round(3)
flag_test_table.columns = ["zero_click_share", "n_eligible"]
print("Flag-linked test -- share of ELIGIBLE rows with prev_30_ctr == 0, by position tier:")
flag_test_table


Flag-linked test -- share of ELIGIBLE rows with prev_30_ctr == 0, by position tier:


,zero_click_share,n_eligible
position_tier,,
top_3,0.093,396
page_1,0.206,3479
striking_distance,0.379,1690
page_2_3,0.413,2523
deep,0.675,77


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Two of the four findings above are already settled (from `w04_baseline_score.ipynb`): position is
a real, strong driver of expected CTR, and volume is too — both support keeping `CTR_BELOW_POSITION_PEERS`
as a live flag rather than shipping a model to replace it yet.

When selecting what contents' snippets to fix, it is better to prioritize the contents that have high positions, as it is common for the contents in low positions to receive no clicks.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

summary = pd.DataFrame({
    "test": ["1. CTR vs position", "2. Volume vs CTR", "3. Staleness vs CTR", "Flag-linked: zero-click share"],
    "flag_linked_to": ["ctr_fix", "quick_win", "refresh", "ctr_fix (this rule's own condition)"],
    "verdict": ["CONFIRMED (see w04_baseline_score.ipynb)", "CONFIRMED (see w04_baseline_score.ipynb)",
                "OPPOSITE/MIXED", "MIXED"],
})
summary


,test,flag_linked_to,verdict
0,1. CTR vs position,ctr_fix,CONFIRMED (see w04_baseline_score.ipynb)
1,2. Volume vs CTR,quick_win,CONFIRMED (see w04_baseline_score.ipynb)
2,3. Staleness vs CTR,refresh,OPPOSITE/MIXED
3,Flag-linked: zero-click share,ctr_fix (this rule's own condition),MIXED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.